In [ ]:
import os
import numpy as np
import pyvista as pv
import matplotlib as mpl
import matplotlib.pyplot as plt

from wakis import SolverFIT3D
from wakis import GridFIT3D
from wakis import WakeSolver
from wakis import geometry

# os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# pv.global_theme.trame.server_proxy_enabled = True
# pv.global_theme.trame.server_proxy_prefix = '/proxy/'
pv.global_theme.window_size = [1000, 600]  # optional

In [ ]:
# # create ring to separate fingers from ttube
ring = pv.ParametricTorus(ringradius=53.0, crosssectionradius=4)
ring = ring.translate((0, 16, 235.65))
ring.save("stl/012_LHC_WVM_6L2-opening.stl")

In [ ]:
stl_body = "stl/012_LHC_WVM_6L2-body.stl"
stl_fingers = "stl/012_LHC_WVM_6L2-fingers.stl"
stl_spring = "stl/012_LHC_WVM_6L2-spring.stl"
stl_ttube = "stl/012_LHC_WVM_6L2-ttube.stl"
stl_opening = "stl/012_LHC_WVM_6L2-opening.stl"
# stl_ttube = 'stl/012_LHC_WVM_6L2-ttube-closed.stl'

geometry.measure_stl_slice(
    [stl_body, stl_fingers, stl_spring, stl_ttube, stl_opening], plane="ZY"
)

In [ ]:
# ---------- Domain setup ---------
surf = (
    pv.read(stl_body) + pv.read(stl_fingers) + pv.read(stl_spring) + pv.read(stl_ttube)
)
# surf=pv.read(stl_ttube)
surf.scale(1e-3, inplace=True)  # scale to meters
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
Lx, Ly, Lz = (xmax - xmin), (ymax - ymin), (zmax - zmin)

# Number of mesh cells
Nx = 250
Ny = 250
Nz = 250

# Readjust for PML
dz0 = (zmax - zmin) / Nz
n_pml = 10
zmin -= dz0 * n_pml
zmax += dz0 * n_pml
Nz += 2 * n_pml

stl_solids = {
    "body": stl_body,
    "fingers": stl_fingers,
    "spring": stl_spring,
    "ttube": stl_ttube,
    "opening": stl_opening,
}

stl_materials = {
    "body": "stainless steel",
    "fingers": "stainless steel",
    "spring": "stainless steel",
    "ttube": "stainless steel",
    "opening": "vacuum",
}

# set grid and geometry
grid = GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    int(Nx),
    int(Ny),
    int(Nz),
    stl_solids=stl_solids,
    stl_materials=stl_materials,
    stl_method="voxelize_rectilinear",
    stl_scale=1e-3,
    subpixel_smoothing=True,
    subpixel_smoothing_factor=4,
    verbose=2,
)

In [ ]:
grid.plot_stl_mask("body")

In [ ]:
# ------------ Beam source ----------------
# Beam parameters
sigmaz = 50e-3  # [m] -> 2 GHz
q = 1e-9  # [C]
beta = 1.0  # beam beta
xs = 0.0  # x source position [m]
ys = 0.0  # y source position [m]
xt = 0.0  # x test position [m]
yt = 0.0  # y test position [m]
# [DEFAULT] tinj = 8.53*sigmaz/c_light  # injection time offset [s]

# Simualtion
wakelength = 30.0  # [m]
add_space = 30  # no. cells

run_folder = ""
results_folder = run_folder + f"012_results_noContactSCS_opening_17M/"
wake = WakeSolver(
    wakelength=wakelength,
    q=q,
    sigmaz=sigmaz,
    beta=beta,
    xsource=xs,
    ysource=ys,
    xtest=xt,
    ytest=yt,
    add_space=add_space,
    results_folder=results_folder,
    Ez_file=results_folder + "Ez.h5",
    verbose=1,
)

In [ ]:
# boundary conditions
bc_low = ["pec", "pec", "pml"]
bc_high = ["pec", "pec", "pml"]

solver = SolverFIT3D(
    grid,
    wake,
    cfln=0.7,
    bc_low=bc_low,
    bc_high=bc_high,
    use_stl=True,
    use_gpu=False,
    # use_sibc=True,
    dtype=np.float32,
    bg="vacuum",
    verbose=2,
)

In [ ]:
# %matplotlib ipympl
solver.sigma.inspect(
    grid=grid, plane="ZY", dpi=200, figsize=(12, 8), backend="pyvista", cmap="Reds_r"
)
# solver.ieps.inspect(plane='ZY', dpi=200, figsize=(12,8), cmap='Reds')

In [ ]:
solver.get_plotting_kwargs("plot2D")
plot_kwargs = {
    "field": "E",
    "component": "Abs",
    "plane": "ZY",
    "pos": 0.5,
    "norm": None,
    "vmin": 0,
    "vmax": 500,
    "figsize": [12, 4],
    "cmap": "plasma",
    "patch_alpha": 0.2,
    "patch_reverse": False,
    "add_patch": ["body", "fingers", "spring", "ttube"],
    "title": results_folder + "Eabs",
    "off_screen": True,
    "interpolation": "spline36",
    "dpi": 200,
    "return_handles": "False",
}

plot_kwargs

In [ ]:
solver.plot2D(
    **plot_kwargs,
)

In [ ]:
# Run wakefield time-domain simulation
solver.wakesolve(
    wakelength=wakelength,
    plot=True,
    plot_func=solver.plot2D,
    plot_every=100,
    plot_until=30000,
    **plot_kwargs,
)

In [ ]:
pl = solver.E.inspect(
    grid=grid,
    plane="YZ",
    dpi=200,
    figsize=(12, 8),
    backend="pyvista",
    cmap="plasma",
    handles=True,
    off_screen=True,
)
# pl.enable_3_lights()

### Plot results

In [ ]:
import numpy as np
from wakis import WakeSolver
from matplotlib import pyplot as plt

wake = WakeSolver(sigmaz=50e-3)
# results_folder = '012_results_pml/'
results_folder = f"012_results_noContact_17M/"
wake.load_results(results_folder)
# wake.solve(skip_cells=70)

In [ ]:
%matplotlib ipympl
# ----------- 1d plot results --------------------
# Plot longitudinal wake potential and impedance
from matplotlib import cm


fig1, ax = plt.subplots(1, 2, figsize=[12, 4], dpi=150)
ax[0].plot(wake.s * 1e3, wake.WP, c="b", lw=2, alpha=0.7, label="Wakis")
ax[0].set_xlabel("s [mm]")
ax[0].set_ylabel("Longitudinal wake potential [V/pC]", color="r")
ax[0].legend()
ax[0].set_xlim(xmax=wake.s[-1] * 1e3)

ax[1].plot(wake.f * 1e-9, np.real(wake.Z), c="b", lw=2, alpha=0.7, label="Re - Wakis")
ax[1].plot(
    wake.f * 1e-9, np.imag(wake.Z), c="b", lw=2, alpha=0.7, ls="--", label="Im - Wakis"
)
ax[1].set_xlabel("f [GHz]")
ax[1].set_ylabel("Longitudinal impedance [Abs][$\Omega$]", color="b")
ax[1].legend()
# ax[1].set_ylim(ymin=0)

# cstWP = wake.read_txt(f'cst/WP_6L2_closed_ttube_allPEC_NOmeshref_120.txt')
# cstZ = wake.read_txt(f'cst/Z_6L2_closed_ttube_allPEC_NOmeshref_120.txt')
cstWP = wake.read_txt(f"cst/WP_nmesh10M_wk3e4.txt")
cstZ = wake.read_txt(f"cst/Z_nmesh10M_wk3e4.txt")
cstZ_2 = wake.read_txt(f"cst/Z_6L2_allPEC_meshref_35.txt")
cstZ_3 = wake.read_txt(f"cst/Z_6L2_allPEC_meshref_40.txt")
cstZ_4 = wake.read_txt(f"cst/Z_6L2_allPEC_meshref_45.txt")

colors = cm.Reds(np.linspace(0.2, 0.8, 4))

ax[0].plot(cstWP[0], cstWP[1], c="r", lw=1.5, label="CST")
# ax[1].plot(cstZ[0], cstZ[1], c=colors[0], lw=1.5, label="Re - CST - 30M mesh")
ax[1].plot(cstZ_2[0], cstZ_2[1], c=colors[1], lw=1.5, label="Re - CST - 30M mesh")
ax[1].plot(cstZ_3[0], cstZ_3[1], c=colors[2], lw=1.5, label="Re - CST - 40M mesh")
ax[1].plot(cstZ_4[0], cstZ_4[1], c=colors[3], lw=1.5, label="Re - CST - 45M mesh")
# ax[1].plot(cstZ[0], cstZ[2], c="k", ls='--', lw=1.5, label="CST")
ax[1].plot(
    cstZ_2[0], cstZ_2[2], c=colors[1], lw=1.5, ls="--", label="Im - CST - 30M mesh"
)
ax[1].plot(
    cstZ_3[0], cstZ_3[2], c=colors[2], lw=1.5, ls="--", label="Im - CST - 40M mesh"
)
ax[1].plot(
    cstZ_4[0], cstZ_4[2], c=colors[3], lw=1.5, ls="--", label="Im - CST - 45M mesh"
)
ax[0].axhline(0, c="grey", ls="--", lw=1)
ax[1].axhline(0, c="grey", ls="--", lw=1)
ax[0].legend()
ax[1].legend()

fig1.tight_layout()
fig1.savefig(results_folder + "001_longitudinal.png")
plt.show()

# # Plot transverse x wake potential and impedance
# fig2, ax = plt.subplots(1, 2, figsize=[12, 4], dpi=150)
# ax[0].plot(wake.s * 1e2, wake.WPx, c="r", lw=1.5, label="Wakis")
# ax[0].set_xlabel("s [cm]")
# ax[0].set_ylabel("Transverse wake potential X [V/pC]", color="r")
# ax[0].legend()
# ax[0].set_xlim(xmax=wake.s[-1] * 1e2)

# ax[1].plot(wake.f * 1e-9, np.abs(wake.Zx), c="b", lw=1.5, label="Wakis")
# ax[1].set_xlabel("f [GHz]")
# ax[1].set_ylabel("Transverse impedance X [Abs][$\Omega$]", color="b")
# ax[1].legend()

# fig2.tight_layout()
# fig2.savefig(results_folder + "001_transverse_x.png")
# plt.show()

# # Plot transverse y wake potential and impedance
# fig3, ax = plt.subplots(1, 2, figsize=[12, 4], dpi=150)
# ax[0].plot(wake.s * 1e2, wake.WPy, c="r", lw=1.5, label="Wakis")
# ax[0].set_xlabel("s [cm]")
# ax[0].set_ylabel("Transverse wake potential Y [V/pC]", color="r")
# ax[0].legend()
# ax[0].set_xlim(xmax=wakelength * 1e2)

# ax[1].plot(wake.f * 1e-9, np.abs(wake.Zy), c="b", lw=1.5, label="Wakis")
# ax[1].set_xlabel("f [GHz]")
# ax[1].set_ylabel("Transverse impedance Y [Abs][$\Omega$]", color="b")
# ax[1].legend()

# fig3.tight_layout()
# fig3.savefig(results_folder + "001_transverse_y.png")
# plt.show()

In [ ]:
!convert -loop 0 -delay 10 012_results_pml/Ez_*.png 012_results_pml/Ez.gif

In [ ]:
# %matplotlib ipympl
%matplotlib inline

import numpy as np
from wakis import WakeSolver
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ----------- Compare multiple result folders --------------------
# Define folders and labels to compare
folders_to_compare = [
    ("012_results_noContact_67M/", "noContact 67M"),
    ("012_results_noContact_28M/", "noContact 28M"),
    ("012_results_noContact_17M/", "noContact 17M"),
    # ('012_results_allContact_28M/', 'allContact 28M'),
]

# Optional CST reference files (set to None to skip)
cst_WP_file = "cst/WP_nmesh10M_wk3e4.txt"
cst_Z_file = "cst/Z_6L2_allPEC_meshref_45.txt"  #'cst/Z_nmesh10M_wk3e4.txt'

skip_cells = 70  # cells to skip at boundaries when solving

# ---------------------------------------------------------------
colors = cm.rainbow(np.linspace(0.1, 0.9, len(folders_to_compare)))

fig, ax = plt.subplots(1, 2, figsize=[14, 5], dpi=150)

for (folder, label), color in zip(folders_to_compare, colors):
    try:
        w = WakeSolver(sigmaz=50e-3)
        w.load_results(folder)
        w.solve(skip_cells=skip_cells)
        ax[0].plot(w.s * 1e3, w.WP, lw=2, alpha=0.7, color=color, label=label)
        ax[1].plot(
            w.f * 1e-9,
            np.real(w.Z),
            lw=2,
            color=color,
            alpha=0.7,
            label=f"Re – {label}",
        )
        # ax[1].plot(w.f * 1e-9, np.imag(w.Z), lw=2, color=color, alpha=0.7,  ls='--', label=f'Im – {label}')
    except Exception as e:
        print(f"[SKIP] {folder}: {e}")

# CST reference
if cst_WP_file and cst_Z_file:
    _w = WakeSolver(sigmaz=50e-3)
    try:
        cstWP = _w.read_txt(cst_WP_file)
        cstZ = _w.read_txt(cst_Z_file)
        ax[0].plot(cstWP[0], cstWP[1], c="k", lw=1.5, label="CST")
        ax[1].plot(cstZ[0], cstZ[1], c="k", lw=1.5, label="Re – CST")
        # ax[1].plot(cstZ[0],  cstZ[2],  c='k', lw=1.5, ls='--', label='Im – CST')
    except Exception as e:
        print(f"[SKIP] CST files: {e}")

ax[0].set_xlabel("s [mm]")
ax[0].set_ylabel("Longitudinal wake potential [V/pC]")
ax[0].legend(fontsize=8)

ax[1].set_xlabel("f [GHz]")
ax[1].set_ylabel("Longitudinal impedance [$\\Omega$]")
ax[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig("012_comparison_skip70.png", dpi=150)
plt.show()

In [ ]:
plt.close("all")